### 2013 ~ 2019

## 2019

In [4]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [5]:
import re
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICDM 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기 (electronic edition via DOI)
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        if doi_tag and doi_tag.get("href"):
            pdf_link = doi_tag["href"]
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [6]:
url = 'https://dblp.org/db/conf/iccv/iccv2019.html'
DB_PATH = "con_db/ICCV_conference_2019.db"
conference_name = 'ICCV 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [7]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICCV_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [9]:
df_papers = get_www_papers('html/ICCV_2019_accepted_papers.html', conference_name)

In [10]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,FaceForensics++: Learning to Detect Manipulate...,"Andreas Rössler, Davide Cozzolino, Luisa Verdo...",https://doi.org/10.1109/ICCV.2019.00009,None,ICCV 2019
1,DeepVCP: An End-to-End Deep Neural Network for...,"Weixin Lu, Guowei Wan, Yao Zhou, Xiangyu Fu, P...",https://doi.org/10.1109/ICCV.2019.00010,None,ICCV 2019
2,Shape Reconstruction Using Differentiable Proj...,"Matheus Gadelha, Rui Wang, Subhransu Maji",https://doi.org/10.1109/ICCV.2019.00011,None,ICCV 2019
3,Fine-Grained Segmentation Networks: Self-Super...,"Måns Larsson, Erik Stenborg, Carl Toft, Lars H...",https://doi.org/10.1109/ICCV.2019.00012,None,ICCV 2019
4,SANet: Scene Agnostic Network for Camera Local...,"Luwei Yang, Ziqian Bai, Chengzhou Tang, Honghu...",https://doi.org/10.1109/ICCV.2019.00013,None,ICCV 2019


In [11]:
save_to_database(df_papers, conference_name, DB_PATH)

1075개의 논문이 ICCV 2019에 저장되었습니다.


# 2017 

In [12]:
url = 'https://dblp.org/db/conf/iccv/iccv2017.html'
DB_PATH = "con_db/ICCV_conference_2017.db"
conference_name = 'ICCV 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [13]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICCV_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [14]:
df_papers = get_www_papers('html/ICCV_2017_accepted_papers.html', conference_name)

In [15]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Globally-Optimal Inlier Set Maximisation for S...,"Dylan Campbell, Lars Petersson, Laurent Kneip,...",https://doi.org/10.1109/ICCV.2017.10,None,ICCV 2017
1,Robust Pseudo Random Fields for Light-Field St...,Chao-Tsung Huang,https://doi.org/10.1109/ICCV.2017.11,None,ICCV 2017
2,A Lightweight Approach for On-the-Fly Reflecta...,"Kihwan Kim, Jinwei Gu, Stephen Tyree, Pavlo Mo...",https://doi.org/10.1109/ICCV.2017.12,None,ICCV 2017
3,Distributed Very Large Scale Bundle Adjustment...,"Runze Zhang, Siyu Zhu, Tian Fang, Long Quan",https://doi.org/10.1109/ICCV.2017.13,None,ICCV 2017
4,Practical Projective Structure from Motion (P2...,"Ludovic Magerand, Alessio Del Bue",https://doi.org/10.1109/ICCV.2017.14,None,ICCV 2017


In [16]:
save_to_database(df_papers, conference_name, DB_PATH)

621개의 논문이 ICCV 2017에 저장되었습니다.


# 2015

In [17]:
url = 'https://dblp.org/db/conf/iccv/iccv2015.html'
DB_PATH = "con_db/ICCV_conference_2015.db"
conference_name = 'ICCV 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [18]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICCV_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [19]:
df_papers = get_www_papers('html/ICCV_2015_accepted_papers.html', conference_name)

In [20]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Ask Your Neurons: A Neural-Based Approach to A...,"Mateusz Malinowski, Marcus Rohrbach, Mario Fritz",https://doi.org/10.1109/ICCV.2015.9,None,ICCV 2015
1,Segment-Phrase Table for Semantic Segmentation...,"Hamid Izadinia, Fereshteh Sadeghi, Santosh Kum...",https://doi.org/10.1109/ICCV.2015.10,None,ICCV 2015
2,Aligning Books and Movies: Towards Story-Like ...,"Yukun Zhu, Ryan Kiros, Richard S. Zemel, Rusla...",https://doi.org/10.1109/ICCV.2015.11,None,ICCV 2015
3,Learning Query and Image Similarities with Ran...,"Ting Yao, Tao Mei, Chong-Wah Ngo",https://doi.org/10.1109/ICCV.2015.12,None,ICCV 2015
4,Learning to See by Moving.,"Pulkit Agrawal, João Carreira, Jitendra Malik",https://doi.org/10.1109/ICCV.2015.13,None,ICCV 2015


In [21]:
save_to_database(df_papers,conference_name, DB_PATH)

526개의 논문이 ICCV 2015에 저장되었습니다.


# 2013

In [22]:
url = 'https://dblp.org/db/conf/iccv/iccv2013.html'
DB_PATH = "con_db/ICCV_conference_2013.db"
conference_name = 'ICCV 2013'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [23]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICCV_2013_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [24]:
df_papers = get_www_papers('html/ICCV_2013_accepted_papers.html', conference_name)

In [25]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,HOGgles: Visualizing Object Detection Features.,"Carl Vondrick, Aditya Khosla, Tomasz Malisiewi...",https://doi.org/10.1109/ICCV.2013.8,None,ICCV 2013
1,How Do You Tell a Blackbird from a Crow?,"Thomas Berg, Peter N. Belhumeur",https://doi.org/10.1109/ICCV.2013.9,None,ICCV 2013
2,Regionlets for Generic Object Detection.,"Xiaoyu Wang, Ming Yang, Shenghuo Zhu, Yuanqing...",https://doi.org/10.1109/ICCV.2013.10,None,ICCV 2013
3,Learning Graphs to Match.,"Minsu Cho, Karteek Alahari, Jean Ponce",https://doi.org/10.1109/ICCV.2013.11,None,ICCV 2013
4,Shape Anchors for Data-Driven Multi-view Recon...,"Andrew Owens, Jianxiong Xiao, Antonio Torralba...",https://doi.org/10.1109/ICCV.2013.461,None,ICCV 2013


In [26]:
save_to_database(df_papers, conference_name, DB_PATH)

454개의 논문이 ICCV 2013에 저장되었습니다.
